# Preparación del dataset con MTCNN

Importación de paquetes

In [1]:
import os
import hashlib
import shutil

import cv2
import numpy as np
from mtcnn import MTCNN
from sklearn.model_selection import train_test_split

Variables del dataset

In [2]:
DATASET_DIR = "dataset"
OUTPUT_DIR = "dataset_processed"
IMG_SIZE = 128
MAX_DIM = 1024
TEST_SIZE = 0.2
RANDOM_STATE = 42
MARGIN = 0.3

CLASSES = sorted(
    d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))
)
print("Clases encontradas:", CLASSES)

Clases encontradas: ['Angry', 'Fear', 'Happy', 'Sad', 'Suprise']


Creación de parejas (ruta, clase) y repartición 80% train 20% test

In [3]:
paths = []
labels = []

for label in CLASSES:
    class_dir = os.path.join(DATASET_DIR, label)
    for fname in sorted(os.listdir(class_dir)):
        paths.append(os.path.join(class_dir, fname))
        labels.append(label)

print(f"Imágenes totales: {len(paths)}")

train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths,
    labels,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=RANDOM_STATE,
)

splits = {
    "train": list(zip(train_paths, train_labels)),
    "test": list(zip(test_paths, test_labels)),
}

for split_name, split_samples in splits.items():
    print(f"{split_name}: {len(split_samples)} imágenes")

Imágenes totales: 59099
train: 47279 imágenes
test: 11820 imágenes


Detección, alineado y recorte con MTCNN

In [4]:
detector = MTCNN()


def align_and_crop(image_rgb, detection, img_size=IMG_SIZE, margin=MARGIN):
    """image_rgb: array HxWx3 en RGB. detection: resultado de detector.detect_faces."""
    keypoints = detection["keypoints"]
    left_eye = np.array(keypoints["left_eye"], dtype=np.float32)
    right_eye = np.array(keypoints["right_eye"], dtype=np.float32)

    # Rotar la imagen completa para alinear los ojos en horizontal
    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))
    eyes_center = tuple((left_eye + right_eye) / 2)

    h, w = image_rgb.shape[:2]
    rot_matrix = cv2.getRotationMatrix2D(eyes_center, angle, 1.0)
    rotated = cv2.warpAffine(image_rgb, rot_matrix, (w, h))

    # Recalcular la caja de la cara en la imagen rotada con un margen
    x, y, box_w, box_h = detection["box"]
    box_points = np.array(
        [[x, y], [x + box_w, y], [x, y + box_h], [x + box_w, y + box_h]],
        dtype=np.float32,
    )
    ones = np.ones((4, 1), dtype=np.float32)
    box_points_h = np.hstack([box_points, ones])
    rotated_box = (rot_matrix @ box_points_h.T).T

    x_min, y_min = rotated_box.min(axis=0)
    x_max, y_max = rotated_box.max(axis=0)

    mx = (x_max - x_min) * margin
    my = (y_max - y_min) * margin
    x_min = max(int(x_min - mx), 0)
    y_min = max(int(y_min - my), 0)
    x_max = min(int(x_max + mx), w)
    y_max = min(int(y_max + my), h)

    if x_max <= x_min or y_max <= y_min:
        return None

    face = rotated[y_min:y_max, x_min:x_max]
    face = cv2.resize(face, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return face

In [5]:
from matplotlib import scale


def process_split(split_name, split_samples):
    failures = {label: 0 for label in CLASSES}
    n_ok = 0

    for fpath, label in split_samples:
        out_dir = os.path.join(OUTPUT_DIR, split_name, label)
        os.makedirs(out_dir, exist_ok=True)

        image_bgr = cv2.imread(fpath)
        if image_bgr is None:
            failures[label] += 1
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        h, w = image_rgb.shape[:2]
        scale = min(1.0, MAX_DIM / max(h, w))
        if scale < 1.0:
            image_rgb = cv2.resize(image_rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

        try:
            detections = detector.detect_faces(image_rgb)
        except Exception:
            failures[label] += 1
            continue
        
        if not detections:
            failures[label] += 1
            continue

        # Si hay varias caras detectadas, se usa la de mayor confianza
        best_detection = max(detections, key=lambda d: d["confidence"])
        face = align_and_crop(image_rgb, best_detection)
        if face is None:
            failures[label] += 1
            continue

        out_name = os.path.basename(fpath)
        out_path = os.path.join(out_dir, out_name)
        face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)
        cv2.imwrite(out_path, face_bgr)
        n_ok += 1

    return n_ok, failures


if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

results = {}
for split_name, split_samples in splits.items():
    n_ok, failures = process_split(split_name, split_samples)
    results[split_name] = {"n_ok": n_ok, "n_total": len(split_samples), "failures": failures}
    print(f"{split_name}: {n_ok}/{len(split_samples)} procesadas correctamente")

train: 39100/47279 procesadas correctamente
test: 9825/11820 procesadas correctamente


## Resumen de fallos de detección de MTCNN (para la memoria)

In [6]:
total_failures = 0
total_images = 0

for split_name, info in results.items():
    print(f"\n--- {split_name} ---")
    for label in CLASSES:
        n_fail = info["failures"][label]
        total_failures += n_fail
        print(f"  {label}: {n_fail} fallos de detección")
    total_images += info["n_total"]

print(f"\nTotal de imágenes: {total_images}")
print(f"Total de fallos de detección MTCNN: {total_failures}")
print(f"Porcentaje de fallo: {100 * total_failures / total_images:.2f}%")


--- train ---
  Angry: 1330 fallos de detección
  Fear: 1600 fallos de detección
  Happy: 1965 fallos de detección
  Sad: 2373 fallos de detección
  Suprise: 911 fallos de detección

--- test ---
  Angry: 306 fallos de detección
  Fear: 398 fallos de detección
  Happy: 471 fallos de detección
  Sad: 603 fallos de detección
  Suprise: 217 fallos de detección

Total de imágenes: 59099
Total de fallos de detección MTCNN: 10174
Porcentaje de fallo: 17.22%
